In [1]:
#%pip install imageio[ffmpeg]

In [2]:
from matplotlib import pyplot as plt
import skimage
import numpy as np
import imageio.v3 as iio
from ipywidgets import interact, IntSlider

In [3]:
focal = 485.82423388827533  # focal length (f)
centerx = 134.875           # principal point x (cx)
centery = 239.875           # principal point y (cy)
ball_radius = 3             # ball radius in centimeters (R)
video_path = 'red_and_blue.mov'      # path to video
num_balls = 2 # number of balls in video

In [4]:
frames = [frame for frame in iio.imiter(video_path)] # load all frames of video

In [5]:
def preprocess_image(image):
    """ Convert image to grayscale and run Canny edge detector.
        Arguments:
            image: RGB image
        Returns:
            Result of Canny edge detector
    """
    return skimage.feature.canny(skimage.color.rgb2gray(image), sigma=3)


In [6]:
def detect_circles(edges, num_balls):
    """ Detect circles in image.
        Arguments:
            edges: edge image produced by preprocess_image()
        Returns:
            List of tuples (x, y, radius)
    """
    hough_radii = np.arange(25,70, 1)
    hough_res = skimage.transform.hough_circle(edges, hough_radii)
    _, cx, cy, radii = skimage.transform.hough_circle_peaks(hough_res, hough_radii, total_num_peaks=num_balls, min_xdistance=80, min_ydistance=80)
    return list(zip(cx, cy, radii))

In [7]:
def calculate_ball_position(x,y,r):
    """ Calculate ball's (X,Y,Z) position in world coordinates
        Arguments:
            x,y: 2D position of ball in image
            r: radius of ball in image
        Returns:
            X,Y,Z position of ball in world coordinates
    """
    Z = focal * ball_radius/ r
    X = (x-centerx)/focal * Z
    Y = (y-centery)/focal * Z
    return (X, Y, Z)

In [8]:
def draw_ball(x,y,r,Z, image, color):
    """ Draw circle on ball and write depth estimate in center
        Arguments:
            x,y,r: 2D position and radius of ball
            Z: estimated depth of ball
    """
    circy, circx = skimage.draw.circle_perimeter(int(y), int(x), int(r), shape=image.shape)
    image[circy, circx] = color
    plt.text(x, y, f"{Z:.1f} cm", color='white', fontsize=10, ha='center', va='center')
    return image

In [9]:
def project(X,Y,Z):
    """ Pinhole projection.
        Arguments:
            X,Y,Z: 3D point
        Returns:    
            (x,y) 2D location of projection in image
    """
    return (focal * (X/Z)+centerx, focal * (Y/Z)+centery)

In [10]:
def draw_line_2d(x1,y1,x2,y2, color):
    """ Draw a 2D line
        Arguments:
            x1,y1: 2D position of first line endpoint
            x2,y2: 2D position of second line endpoint
    """
    plt.plot([x1, x2], [y1, y2], color=color, linewidth=1)

In [11]:
def draw_line_3d(X1,Y1,Z1,X2,Y2,Z2, color):
    """ Draw a 3D line.  (This should call draw_line_2d.)
        Arguments:
            X1,Y1,Z1: 3D position of first line endpoint
            X2,Y2,Z2: 3D position of second line endpoint
    """
    x1, y1 = project(X1, Y1, Z1)
    x2, y2 = project(X2, Y2, Z2)
    draw_line_2d(x1, y1,x2, y2, color)

In [12]:
def draw_bounding_cube(X,Y,Z, color):
    """ Draw bounding cube around 3D point, with radius R
        Arguments:
            image: image on which to draw
            X,Y,Z: 3D center point of cube
    """
    R=ball_radius
    draw_line_3d(X-R, Y-R, Z-R, X+R, Y-R, Z-R, color)
    draw_line_3d(X+R, Y-R, Z-R, X+R, Y+R, Z-R, color)
    draw_line_3d(X+R, Y+R, Z-R, X-R, Y+R, Z-R, color)
    draw_line_3d(X-R, Y+R, Z-R, X-R, Y-R, Z-R, color)
    draw_line_3d(X-R, Y-R, Z+R, X+R, Y-R, Z+R, color)
    draw_line_3d(X+R, Y-R, Z+R, X+R, Y+R, Z+R, color)
    draw_line_3d(X+R, Y+R, Z+R, X-R, Y+R, Z+R, color)
    draw_line_3d(X-R, Y+R, Z+R, X-R, Y-R, Z+R, color)
    draw_line_3d(X-R, Y-R, Z-R, X-R, Y-R, Z+R, color)
    draw_line_3d(X+R, Y-R, Z-R, X+R, Y-R, Z+R, color)
    draw_line_3d(X+R, Y+R, Z-R, X+R, Y+R, Z+R, color)
    draw_line_3d(X-R, Y+R, Z-R, X-R, Y+R, Z+R, color)

In [ ]:
# example of interactive video display
def process_frame(i):
    edges = preprocess_image(frames[i])
    balls = detect_circles(edges, num_balls)
    plt.imshow(frames[i])
    for x, y, r in balls:
        X, Y, Z = calculate_ball_position(x, y, r)
        draw_ball(x, y, r, Z,frames[i], (0, 255, 0))
        draw_bounding_cube(X,Y,Z, "magenta")
interact(process_frame, i=IntSlider(min=0, max=len(frames)-1, step=1, value=0))

interactive(children=(IntSlider(value=0, description='i', max=338), Output()), _dom_classes=('widget-interact'…

<function __main__.process_frame(i)>

In [14]:
Xs, Ys, Zs = [], [], []

for frame in frames:
    edges = preprocess_image(frame)
    balls = detect_circles(edges, num_balls)
    for x, y, r in balls:
        X, Y, Z = calculate_ball_position(x, y, r)
        Xs.append(X)
        Ys.append(Y)
        Zs.append(Z)

fig = plt.figure()
ax = fig.add_subplot(projection='3d')
ax.scatter(Xs, Ys, Zs)
ax.set_xlabel('X (cm)')
ax.set_ylabel('Y (cm)')
ax.set_zlabel('Z (cm)')
plt.show()

KeyboardInterrupt: 